# Study 902 — Multi-Factor Composite — the teardown

The excess-of-cash Sharpe race, the HAC *t* on the active return, the paired moving-block bootstrap on the Sharpe advantage, the two-era robustness cut, the diversification decomposition, the per-sleeve breakdown, the inverse-vol robustness alt, and the planted-edge synthetic control. Real numbers are quoted from the frozen `R` dict (`examples/verify.py` reproduces them from the cache); the live cell runs the synthetic control offline.

In [1]:
R = {'start': '2013-08-31', 'end': '2026-06-30', 'n_months': 155, 'fingerprint': '8b05ab0c64b8', 'turnover_pct': 1.17, 'cost_bps_yr': 0.3, 'comp_cagr': 13.32, 'comp_vol': 14.1, 'comp_sharpe': 0.841, 'comp_maxdd': -23.7, 'spy_cagr': 14.14, 'spy_vol': 14.5, 'spy_sharpe': 0.874, 'spy_maxdd': -23.9, 'adv': -0.033, 'active_bps': -6.5, 't_active': -0.73, 'win_rate': 51, 'boot_lo': -0.205, 'boot_hi': 0.077, 'boot_pneg': 0.804, 'cbb_lo': 0.389, 'cbb_hi': 1.373, 'era_early_adv': 0.078, 'era_early_t': -0.02, 'era_early_n': 77, 'era_late_adv': -0.085, 'era_late_t': -0.77, 'era_late_n': 78, 'mean_single_vol': 15.2, 'min_single_vol': 11.6, 'mean_single_sharpe': 0.786, 'best_single_sharpe': 0.928, 'cross_disp_pp': 7.0, 'blend_annual_sd': 11.9, 'single': {'VLUE': 0.699, 'QUAL': 0.841, 'MTUM': 0.928, 'USMV': 0.78, 'SIZE': 0.681}, 'invvol_sharpe': 0.832, 'invvol_t': -1.3, 'null_adv': 0.007, 'null_t': 0.21, 'planted_adv': 0.198, 'planted_t': 3.66}
print('window %s -> %s  (%d months, fingerprint %s)'
      % (R['start'], R['end'], R['n_months'], R['fingerprint']))

window 2013-08-31 -> 2026-06-30  (155 months, fingerprint 8b05ab0c64b8)


## 1. The excess-of-cash Sharpe race (both legs minus BIL)

The core comparison. Composite **net** of a one-way 2 bps rebalance cost vs SPY, both in excess of the tradable BIL T-bill ETF.

In [2]:
print('leg           exSharpe   CAGR     vol     maxDD')
print('composite net %8.3f  %6.2f%%  %5.1f%%  %6.1f%%'
      % (R['comp_sharpe'], R['comp_cagr'], R['comp_vol'], R['comp_maxdd']))
print('SPY           %8.3f  %6.2f%%  %5.1f%%  %6.1f%%'
      % (R['spy_sharpe'], R['spy_cagr'], R['spy_vol'], R['spy_maxdd']))
print()
print('Sharpe advantage (comp - SPY): %+.3f' % R['adv'])
print('active return: %+.1f bps/mo   NW t = %+.2f   win-rate %d%%'
      % (R['active_bps'], R['t_active'], R['win_rate']))

leg           exSharpe   CAGR     vol     maxDD
composite net    0.841   13.32%   14.1%   -23.7%
SPY              0.874   14.14%   14.5%   -23.9%

Sharpe advantage (comp - SPY): -0.033
active return: -6.5 bps/mo   NW t = -0.73   win-rate 51%


## 2. Bootstrap CI on the advantage + two-era robustness

A green Signal needs the advantage clear of zero *and* stable across sub-eras. Neither holds: the paired moving-block bootstrap CI straddles zero, and the advantage flips sign from the first half to the second.

In [3]:
print('paired block bootstrap on Sharpe advantage:')
print('  adv %+.3f   95%% CI [%+.3f, %+.3f]   P(adv<0) = %.2f'
      % (R['adv'], R['boot_lo'], R['boot_hi'], R['boot_pneg']))
print('  (composite excess Sharpe CBB CI [%.3f, %.3f] — wide on 155 months)'
      % (R['cbb_lo'], R['cbb_hi']))
print()
print('two-era split:')
print('  early (%dm): adv %+.3f   active NW t %+.2f'
      % (R['era_early_n'], R['era_early_adv'], R['era_early_t']))
print('  late  (%dm): adv %+.3f   active NW t %+.2f  <- sign flip'
      % (R['era_late_n'], R['era_late_adv'], R['era_late_t']))

paired block bootstrap on Sharpe advantage:
  adv -0.033   95% CI [-0.205, +0.077]   P(adv<0) = 0.80
  (composite excess Sharpe CBB CI [0.389, 1.373] — wide on 155 months)

two-era split:
  early (77m): adv +0.078   active NW t -0.02
  late  (78m): adv -0.085   active NW t -0.77  <- sign flip


## 3. What the blend DOES diversify — the factor-timing pitch

The diversification claim is not empty: the blend genuinely beats the *average* single sleeve on both vol and Sharpe, and it dampens the wide cross-sleeve dispersion each year. It just doesn't clear the *market*.

In [4]:
print('composite vol %.1f%% < mean single sleeve %.1f%% (min single %.1f%%, SPY %.1f%%)'
      % (R['comp_vol'], R['mean_single_vol'], R['min_single_vol'], R['spy_vol']))
print('composite Sharpe %.3f > mean single %.3f (best single %.3f, SPY %.3f)'
      % (R['comp_sharpe'], R['mean_single_sharpe'], R['best_single_sharpe'], R['spy_sharpe']))
print('avg cross-sleeve annual dispersion %.1f pp -> blend year-to-year sd %.1f pp'
      % (R['cross_disp_pp'], R['blend_annual_sd']))
print()
print('per single-factor sleeve excess Sharpe (common window):')
for tk, s in R['single'].items():
    print('  %-5s %.3f' % (tk, s))

composite vol 14.1% < mean single sleeve 15.2% (min single 11.6%, SPY 14.5%)
composite Sharpe 0.841 > mean single 0.786 (best single 0.928, SPY 0.874)
avg cross-sleeve annual dispersion 7.0 pp -> blend year-to-year sd 11.9 pp

per single-factor sleeve excess Sharpe (common window):
  VLUE  0.699
  QUAL  0.841
  MTUM  0.928
  USMV  0.780
  SIZE  0.681


## 4. Robustness — inverse-vol weighting

Risk-weighting the sleeve (inverse trailing 12-month vol, point-in-time) doesn't rescue it: the advantage stays negative.

In [5]:
print('inverse-vol sleeve net exSharpe %.3f vs SPY %.3f (adv %+.3f, active NW t %+.2f)'
      % (R['invvol_sharpe'], R['spy_sharpe'], R['invvol_sharpe']-R['spy_sharpe'], R['invvol_t']))

inverse-vol sleeve net exSharpe 0.832 vs SPY 0.874 (adv -0.042, active NW t -1.30)


## 5. Synthetic control — the machinery is faithful

The detector recovers a *planted* per-annum blend edge and stays silent on the null. Machinery proof only — never cited in support of a stamp.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from multi_factor import data, strategy as st
for label, edge in [('null    (edge=+0%/yr)', 0.0), ('planted (edge=+3%/yr)', 0.03)]:
    d = st.synthetic_detect(data.synthetic_world(n_months=168, edge_ann=edge, seed=902))
    print('%s: Sharpe adv %+.3f  active %+.1f bps/mo  NW t = %+.2f'
          % (label, d['sharpe_adv'], d['active_bps'], d['t_active_nw']))

null    (edge=+0%/yr): Sharpe adv +0.007  active +1.5 bps/mo  NW t = +0.21
planted (edge=+3%/yr): Sharpe adv +0.198  active +26.5 bps/mo  NW t = +3.66


## Verdict

**Signal — Weak.** The diversification the sleeve is sold on is **real and mechanical**: the equal-weight blend carries lower vol (14.1%) and a higher excess Sharpe (0.841) than the *average* single factor sleeve (15.2% / 0.786), and it tames the ~7 pp/yr cross-sleeve dispersion. But the headline claim — *beat the market on risk-adjusted return* — fails: the excess-of-cash Sharpe advantage over SPY is **-0.033** (active NW *t* **-0.73**), the bootstrap CI **[-0.205, +0.077]** straddles zero, and it flips sign across eras. No SPY-beating edge; a genuine diversification benefit only. Flagship-survivor selection is named.

**Tradability — Fragile.** Costs are *not* the obstacle — the sleeve turns over 1.2% of NAV/mo (0.3 bps/yr), so the gross and net numbers are identical to two decimals. The real, deliverable benefit (diversification of factor-timing risk, one ticket per sleeve, penny spreads) is trivially buyable — but what it buys is a marginally-lower-Sharpe, lower-vol clone of SPY, not a bankable market-beating edge. Real but thin → Fragile, not Investable.